# Void Shadow Studio ? Colab GPU render

In Colab, choose **Runtime ? Change runtime type ? GPU**, then **Runtime ? Run all**. A Google account and an available GPU are required. This text-to-video notebook runs in Colab and downloads an MP4 when it finishes. Upload that MP4 on the scene page in the studio.

In [ ]:
from pathlib import Path
import subprocess
import sys
import torch

if not Path("/content").is_dir() or not torch.cuda.is_available():
    raise RuntimeError("Open this notebook in Google Colab and select a GPU runtime.")
if torch.cuda.get_device_properties(0).total_memory < 12 * 1024**3:
    raise RuntimeError("Select a GPU runtime with at least 12 GiB of GPU memory.")

deps = Path("/tmp/void-colab-dependencies")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "--target", str(deps),
    "diffusers==0.35.2", "transformers==4.51.3", "accelerate==1.7.0",
    "huggingface-hub==0.36.2", "tokenizers>=0.21,<0.22", "safetensors>=0.4.3,<1",
    "Pillow>=10,<13", "sentencepiece", "ftfy", "imageio", "imageio-ffmpeg", "importlib-metadata", "zipp",
])
sys.path.insert(0, str(deps))

In [ ]:
SCENE = {"scene_id": 0, "prompt": "A quiet railway at dawn.", "opening_prompt": "", "negative_prompt": "", "seed": 42}

In [ ]:
import gc
import numpy as np
from diffusers import CogVideoXDPMScheduler, CogVideoXPipeline
from transformers import T5EncoderModel, T5Tokenizer

MODEL = "zai-org/CogVideoX-2b"
DTYPE = torch.float16
GPU = torch.device("cuda:0")
shots = [(SCENE["opening_prompt"], 25, 18)] if SCENE["opening_prompt"] else []
shots.append((SCENE["prompt"], 49, 36))

print("Encoding the scene prompts.", flush=True)
tokenizer = T5Tokenizer.from_pretrained(MODEL, subfolder="tokenizer")
encoder = T5EncoderModel.from_pretrained(
    MODEL, subfolder="text_encoder", torch_dtype=DTYPE,
    low_cpu_mem_usage=True, use_safetensors=True, device_map={"": 0},
)
tokens = tokenizer([shot[0] for shot in shots] + [SCENE["negative_prompt"]],
                   padding="max_length", max_length=226, truncation=True, return_tensors="pt")
with torch.inference_mode():
    embeddings = encoder(tokens.input_ids.to(GPU))[0].to(dtype=DTYPE, device="cpu")
del encoder, tokenizer, tokens
gc.collect()
torch.cuda.empty_cache()

print("Loading the video model.", flush=True)
pipe = CogVideoXPipeline.from_pretrained(
    MODEL, text_encoder=None, tokenizer=None, torch_dtype=DTYPE, low_cpu_mem_usage=True, use_safetensors=True,
)
pipe.scheduler = CogVideoXDPMScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")
pipe.enable_model_cpu_offload(gpu_id=0)
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

clips = []
for index, (_, frames, steps) in enumerate(shots):
    print(f"Generating shot {index + 1} of {len(shots)}.", flush=True)
    with torch.inference_mode():
        clip = pipe(
            prompt_embeds=embeddings[index:index + 1].to(GPU),
            negative_prompt_embeds=embeddings[-1:].to(GPU),
            width=720, height=480, num_frames=frames, num_inference_steps=steps,
            guidance_scale=6.0, use_dynamic_cfg=True, output_type="np",
            generator=torch.Generator(device="cpu").manual_seed((SCENE["seed"] + index) % 2**32),
        ).frames[0]
    if len(clip) != frames or not np.isfinite(clip).all():
        raise RuntimeError("The GPU returned invalid video frames.")
    clips.append(clip)

In [ ]:
from diffusers.utils import export_to_video
from google.colab import files
import imageio_ffmpeg

source = Path("/content/void-studio-source.mp4")
output = Path(f'/content/scene-{SCENE["scene_id"]}-colab-1080p.mp4')
export_to_video(np.concatenate(clips, axis=0), str(source), fps=8)
subprocess.check_call([
    imageio_ffmpeg.get_ffmpeg_exe(), "-hide_banner", "-loglevel", "error", "-y",
    "-i", str(source), "-vf", "scale=1620:1080:flags=lanczos,pad=1920:1080:150:0,setsar=1",
    "-c:v", "libx264", "-preset", "fast", "-crf", "20", "-pix_fmt", "yuv420p",
    "-an", "-movflags", "+faststart", str(output),
])
print(f"Exported {output.name}. Upload it on the scene page.")
files.download(str(output))